#### load data

In [9]:
import io
import pandas as pd
import requests
# if 'data_loader' not in globals():
#     from mage_ai.data_preparation.decorators import data_loader
# if 'test' not in globals():
#     from mage_ai.data_preparation.decorators import test


source_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet'


# @data_loader
def load_data_from_api(*args, **kwargs):

    df = pd.read_parquet(source_url)
    print(f'number of records: {df.shape[0]}')
    return df

df = load_data_from_api()

number of records: 3403766


In [10]:
type(df)

pandas.core.frame.DataFrame

#### feature engineering

In [11]:
def read_dataframe(data, *args, **kwargs):
    
    df = data.copy()
    print(f'Preprocessing data with {df.shape[0]} records')

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    print(f'number of records: {df.shape[0]}')

    return df

df = read_dataframe(df)


Preprocessing data with 3403766 records
number of records: 3316216


In [13]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee', 'duration'],
      dtype='object')

In [19]:
import os
import pickle
import click
import mlflow

from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

from sklearn.feature_extraction import DictVectorizer


In [28]:

# HPO_EXPERIMENT_NAME = "hw3_taxi"
EXPERIMENT_NAME = "hw3_taxi_lr"
RF_PARAMS = []

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()

In [29]:
mlflow.get_tracking_uri()

# Get a list of all experiments
all_experiments = mlflow.search_experiments()

for exp in all_experiments:
    print(f"ID: {exp.experiment_id}, Name: {exp.name}")

ID: 5, Name: hw3_taxi_lr
ID: 4, Name: homework_3
ID: 3, Name: random-forest-best-models
ID: 2, Name: random-forest-hyperopt
ID: 1, Name: homework_2
ID: 0, Name: Default


In [ ]:
to_encode = ['PULocationID', 'DOLocationID']
df[to_encode] = df[to_encode].astype(str)
dicts = df[to_encode].to_dict(orient='records')

dv = DictVectorizer()
X = dv.fit_transform(dicts)
y = df.duration


LinearRegression()

In [33]:
with mlflow.start_run():
    lr = LinearRegression()
    lr.fit(X, y)
    # Evaluate model on the validation and test sets
    train_rmse = root_mean_squared_error(y, lr.predict(X))#, squared=False)
    mlflow.log_metric("train_rmse", train_rmse)

2025/06/09 23:31:44 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/06/09 23:33:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: RESOURCE_DOES_NOT_EXIST: Run with id=dff70f14dd0649eb95a9e65c094ae75b not found


RestException: RESOURCE_DOES_NOT_EXIST: Run with id=dff70f14dd0649eb95a9e65c094ae75b not found

In [34]:

client = MlflowClient()


In [36]:

# Retrieve the top_n model runs and log the models
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
experiment


<Experiment: artifact_location='/Users/matthiasmotl/.Trash/02_experiment_tracking/homework/artifacts/5', creation_time=1749503421776, experiment_id='5', last_update_time=1749503421776, lifecycle_stage='active', name='hw3_taxi_lr', tags={}>

In [39]:

runs = client.search_runs(
    experiment_ids=experiment.experiment_id,
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"])
        


In [42]:
# Step 1: Tell MLflow where your server is (crucial for connecting)
mlflow.set_tracking_uri("http://127.0.0:5000")

# Step 2: Tell MLflow which experiment to use for all following runs
# This will either find the existing experiment or create a new one.
mlflow.set_experiment("hw3_taxi_lr")

# Step 3: Now run your code. It will automatically be logged to "hw3_taxi_lr".
with mlflow.start_run(run_name="Linear Regression Baseline"): # Giving the run a name is good practice
    # Your code is perfect here
    lr = LinearRegression()
    lr.fit(X, y)

    train_rmse = root_mean_squared_error(y, lr.predict(X))
    print(f"Train RMSE: {train_rmse}")
    
    # --- Logging ---
    
    # Log the metric
    mlflow.log_metric("train_rmse", train_rmse)
    
    # BEST PRACTICE: Also log parameters and the model itself!
    mlflow.log_params(lr.get_params())
    mlflow.sklearn.log_model(lr, artifact_path="model")
    
    print("Run logged successfully!")

KeyboardInterrupt: 

In [40]:
runs

[<Run: data=<RunData: metrics={}, params={}, tags={'mlflow.runName': 'omniscient-snail-624',
  'mlflow.source.name': '/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',
  'mlflow.user': 'matthiasmotl'}>, info=<RunInfo: artifact_uri='/Users/matthiasmotl/.Trash/02_experiment_tracking/homework/artifacts/5/dff70f14dd0649eb95a9e65c094ae75b/artifacts', end_time=None, experiment_id='5', lifecycle_stage='active', run_id='dff70f14dd0649eb95a9e65c094ae75b', run_name='omniscient-snail-624', run_uuid='dff70f14dd0649eb95a9e65c094ae75b', start_time=1749504686618, status='RUNNING', user_id='matthiasmotl'>, inputs=<RunInputs: dataset_inputs=[]>>,
 <Run: data=<RunData: metrics={}, params={}, tags={'mlflow.runName': 'Linear Regression Baseline',
  'mlflow.source.name': '/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',


In [41]:
mlflow.register_model(
    "runs:/"+runs[0].info.run_id+"/model",
    "hw3_taxi_lr_model"
)

Successfully registered model 'hw3_taxi_lr_model'.


RestException: RESOURCE_DOES_NOT_EXIST: Run with id=dff70f14dd0649eb95a9e65c094ae75b not found

In [ ]:
    )
    for run in runs:
        train_and_log_model(data_path=data_path, params=run.data.params)

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    best_run = client.search_runs(
        experiment_ids=2, #this is random-forest-hyperopt,
        filter_string='', #'metrics.rmse < 5.36',
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=1,
        order_by=['metrics.rmse ASC'])[0]

    # Register the best model
    # mlflow.register_model( ... )
    model_uri = f'runs:/{best_run}/model'
    mlflow.register_model(model_uri=model_uri, name='best_hw2_randomforest')


if __name__ == '__main__':
    run_register_model()
